# Debug da Jornada do Usuário — Sock Shop

Cada célula representa uma etapa isolada do script Locust do ConfigMap.
Execute célula a célula para inspecionar request e response de cada passo.

**Problema identificado:** `POST /addresses` enviava `userID` vazio porque
`req.session.customerId` não estava disponível — race condition entre o
front-end salvar a sessão no Redis e a próxima requisição carregar essa sessão.

**Correções aplicadas:**
- `sleep(1)` após register e login para garantir que o Redis persista a sessão
- Verificação de consistência dos cookies (`md.sid` ↔ `logged_in`)
- `is_ok_response()` para detectar corretamente o body `{"status_code": 500}`
  que o front-end retorna com HTTP 200 quando endereço/cartão não existe
- Diagnóstico imediato após criar endereço/cartão

**Fluxo:**
1. Imports
2. Configuração
3. Helpers
4. Registro + sleep(1)
5. Login + sleep(1) + verificação de sessão
6. Página inicial
7. Categoria
8. Catálogo (escolhe produto)
9. Limpar carrinho
10. Adicionar ao carrinho
11. Ver carrinho
12. Criar endereço + diagnóstico imediato
13. Criar cartão + diagnóstico imediato
14. Diagnóstico completo de sessão
15. Consultar endereço/cartão padrão
16. Finalizar pedido
17. Ver pedidos
18. Estado final da sessão

In [1]:
# ============================================================
# CÉLULA 1 — Imports
# ============================================================

import requests
import base64
import re
import random
import json
from uuid import uuid4
from pprint import pprint
from time import sleep
from urllib.parse import unquote

print("Imports OK")

Imports OK


In [2]:
# ============================================================
# CÉLULA 2 — Configuração
# ============================================================
# IMPORTANTE: Use kubectl port-forward para garantir session affinity
# (todas as requisições vão para o MESMO pod do front-end).
#
# Em OUTRO terminal, execute:
#   kubectl port-forward -n sock-shop svc/front-end 8080:80
#
# Depois volte aqui e use localhost:8080

BASE_URL = "http://localhost:8080"  # ← porta-forward (garante sticky session)
TIMEOUT  = 15                        # segundos por requisição

# Sessão HTTP compartilhada entre todas as células (preserva cookies)
session = requests.Session()

# Estado mutable do usuário virtual
state = {
    "username":    None,
    "password":    None,
    "email":       None,
    "customer_id": None,
    "item_id":     None,
    "address_id":  None,
    "card_id":     None,
}

print(f"BASE_URL : {BASE_URL}")
print(f"Estado   : {state}")
print("\n⚠ IMPORTANTE: Se receber erro de conexão, execute em outro terminal:")
print("  kubectl port-forward -n sock-shop svc/front-end 8080:80")

BASE_URL : http://localhost:8080
Estado   : {'username': None, 'password': None, 'email': None, 'customer_id': None, 'item_id': None, 'address_id': None, 'card_id': None}

⚠ IMPORTANTE: Se receber erro de conexão, execute em outro terminal:
  kubectl port-forward -n sock-shop svc/front-end 8080:80


In [3]:
# ============================================================
# CÉLULA 3 — Helpers
# ============================================================

def is_valid_object_id(value):
    """Valida formato de Mongo ObjectId (24 hex chars)."""
    return isinstance(value, str) and bool(re.fullmatch(r"[a-fA-F0-9]{24}", value))


def is_ok_response(resp):
    """
    Retorna True apenas se HTTP 200 E o body NÃO é {"status_code": 500}.

    O front-end retorna HTTP 200 com {"status_code": 500} quando o recurso
    não existe (endereço ou cartão), então checar só o status HTTP é insuficiente.
    """
    if resp.status_code != 200:
        return False
    try:
        data = resp.json()
        if isinstance(data, dict) and data.get("status_code") == 500:
            return False
    except Exception:
        pass
    return True


def response_has_app_error(resp):
    try:
        data = resp.json()
        return isinstance(data, dict) and bool(data.get("error"))
    except Exception:
        return False


def extract_object_id(data, link_rel=None):
    """Extrai ObjectId de respostas diretas ou _links HAL."""
    if not isinstance(data, dict):
        return None
    candidate = data.get("id") or data.get("_id")
    if is_valid_object_id(candidate):
        return candidate
    links = data.get("_links")
    if isinstance(links, dict) and link_rel:
        rel = links.get(link_rel)
        if isinstance(rel, dict):
            href = rel.get("href", "")
            maybe_id = href.rstrip("/").split("/")[-1]
            if is_valid_object_id(maybe_id):
                return maybe_id
    return None


def show(resp):
    """Exibe status, headers e body de forma legível."""
    print(f"STATUS  : {resp.status_code}")
    print("HEADERS :")
    pprint(dict(resp.headers))
    print("\nBODY    :")
    try:
        pprint(resp.json())
    except Exception:
        print(resp.text[:2000])


def check_session_cookies():
    """
    Verifica se md.sid e logged_in são consistentes.

    O front-end (helpers/index.js) usa req.session.customerId quando
    logged_in existe. Se md.sid aponta para uma sessão diferente da que
    foi autenticada, customerId estará ausente e POST /addresses enviará
    userID vazio — gerando endereços órfãos sem dono.
    """
    md_sid_raw  = session.cookies.get('md.sid', '')
    logged_in   = session.cookies.get('logged_in', '')

    print(f"md.sid (raw) : {md_sid_raw!r}")
    print(f"logged_in    : {logged_in!r}")

    if not md_sid_raw:
        print("⚠ Cookie md.sid ausente — sessão não foi criada!")
        return False
    if not logged_in:
        print("⚠ Cookie logged_in ausente — usuário não está logado!")
        return False

    # Express-session assina o cookie como "s:SESSION_ID.SIGNATURE"
    decoded = unquote(md_sid_raw)
    if decoded.startswith('s:'):
        session_id = decoded[2:].split('.')[0]
        print(f"session ID   : {session_id!r}")
        if session_id == logged_in:
            print("✓ md.sid e logged_in consistentes — customerId estará na sessão")
            return True
        else:
            print("✗ PROBLEMA: md.sid e logged_in NÃO correspondem!")
            print("  → customerId não estará na sessão ativa")
            print("  → POST /addresses vai criar endereço SEM dono (userID vazio)")
            return False
    else:
        print(f"  decoded md.sid: {decoded!r}")
        return False


print("Helpers definidos OK")

Helpers definidos OK


In [4]:
# ============================================================
# CÉLULA 4 — Gerar identidade + Registro  (POST /register)
# ============================================================

# Gera credenciais únicas para este usuário virtual
state["username"]    = f"user_{uuid4().hex[:8]}"
state["password"]    = "Passw0rd!"
state["email"]       = f"{state['username']}@example.com"
state["customer_id"] = None

print(f"username : {state['username']}")
print(f"password : {state['password']}")
print(f"email    : {state['email']}\n")

payload = {
    "username": state["username"],
    "password": state["password"],
    "email":    state["email"],
}

resp = session.post(f"{BASE_URL}/register", json=payload, timeout=TIMEOUT)
show(resp)

if resp.status_code in (200, 201):
    data = resp.json()
    created_id = data.get("id") or data.get("_id")
    if is_valid_object_id(created_id):
        state["customer_id"] = created_id
    print(f"\n✓ Registro OK — customer_id: {state['customer_id']}")

    # Aguarda o express-session salvar a sessão no Redis antes da próxima
    # requisição. Sem este sleep, POST /addresses pode chegar ao servidor
    # antes que o Redis tenha persistido req.session.customerId, resultando
    # em userID vazio e endereços criados sem dono.
    sleep(1)
    print("  (sessão salva no Redis)")
else:
    print("\n✗ Registro falhou")

username : user_fdc45c1b
password : Passw0rd!
email    : user_fdc45c1b@example.com

STATUS  : 200
HEADERS :
{'Connection': 'keep-alive',
 'Content-Length': '33',
 'Content-Type': 'application/json; charset=utf-8',
 'Date': 'Sun, 26 Apr 2026 13:58:24 GMT',
 'ETag': 'W/"21-BmayCoflCujjswKFcrnd65zrJoE"',
 'X-Powered-By': 'Express',
 'set-cookie': 'logged_in=dRAW0Bp9kmhh7zJtGQYq4NCWKqARdEt3; Max-Age=3600; '
               'Path=/; Expires=Sun, 26 Apr 2026 14:58:24 GMT, '
               'md.sid=s%3AdRAW0Bp9kmhh7zJtGQYq4NCWKqARdEt3.DDzf5i6RoNXdoGRmrak25AVYBCxDhtx6j6PMrPzaPUk; '
               'Path=/; HttpOnly'}

BODY    :
{'id': '69ee1a004a863f0001768b02'}

✓ Registro OK — customer_id: 69ee1a004a863f0001768b02
  (sessão salva no Redis)


In [5]:
# ============================================================
# CÉLULA 5 — Login  (GET /login) + verificação de sessão
# ============================================================

auth_header = base64.b64encode(
    f"{state['username']}:{state['password']}".encode("utf-8")
).decode("ascii")

resp = session.get(
    f"{BASE_URL}/login",
    headers={
        "Authorization":    f"Basic {auth_header}",
        "X-Requested-With": "XMLHttpRequest",
    },
    timeout=TIMEOUT,
)
show(resp)

if resp.status_code == 200:
    # O front-end retorna "Cookie is set" (texto), não JSON.
    # O customer_id já foi capturado no registro; o login apenas
    # re-associa o customerId na sessão e renova o cookie logged_in.
    print(f"\n✓ Login OK — customer_id mantido: {state['customer_id']}")

    # Mesmo sleep aplicado após register: garante que req.session.customerId
    # está persistido no Redis antes de qualquer chamada subsequente.
    sleep(1)
    print("  (sessão salva no Redis)\n")

    print("── Verificação de cookies ──")
    ok = check_session_cookies()
    if not ok:
        print("\n⚠ Sessão inconsistente — as próximas células podem falhar.")
        print("  Reinicie o kernel, execute as células 1-5 novamente e verifique.")
else:
    print("\n✗ Login falhou")

STATUS  : 200
HEADERS :
{'Connection': 'keep-alive',
 'Content-Length': '13',
 'Content-Type': 'text/html; charset=utf-8',
 'Date': 'Sun, 26 Apr 2026 13:58:25 GMT',
 'ETag': 'W/"d-OaJpCA6TxxBt1wHxeVRVz2LhB2I"',
 'X-Powered-By': 'Express',
 'set-cookie': 'logged_in=SmPehyI1H8pB01injoAAe2f5qRA6VFe6; Max-Age=3600; '
               'Path=/; Expires=Sun, 26 Apr 2026 14:58:25 GMT, '
               'md.sid=s%3ASmPehyI1H8pB01injoAAe2f5qRA6VFe6.4tyW1ES%2FssvqCEuLDcSe90xkyLdl%2FYKJBj0fk1ucsGw; '
               'Path=/; HttpOnly'}

BODY    :
Cookie is set

✓ Login OK — customer_id mantido: 69ee1a004a863f0001768b02
  (sessão salva no Redis)

── Verificação de cookies ──
md.sid (raw) : 's%3ASmPehyI1H8pB01injoAAe2f5qRA6VFe6.4tyW1ES%2FssvqCEuLDcSe90xkyLdl%2FYKJBj0fk1ucsGw'
logged_in    : 'SmPehyI1H8pB01injoAAe2f5qRA6VFe6'
session ID   : 'SmPehyI1H8pB01injoAAe2f5qRA6VFe6'
✓ md.sid e logged_in consistentes — customerId estará na sessão


## ⚠️ Problema: Session Affinity com NodePort

Se estiver usando IP direto do NodePort (ex: `http://100.73.36.16:8080`), o servidor renova o `md.sid` a cada POST sem sincronizar `logged_in`.

**Solução:** Use `kubectl port-forward` em outro terminal:

```bash
kubectl port-forward -n sock-shop svc/front-end 8080:80
```

Depois altere a Célula 2 para:
```python
BASE_URL = "http://localhost:8080"
```

Isso garante que **todas as requisições vão para o MESMO pod**, preservando a sessão.

Execute a Célula 14 após as alterações para confirmar que `md.sid` e `logged_in` ficam consistentes.

In [6]:
# ============================================================
# CÉLULA 6 — Página inicial  (GET /)
# ============================================================

resp = session.get(f"{BASE_URL}/", timeout=TIMEOUT)
show(resp)
print(f"\n✓ Página inicial: {resp.status_code}")

STATUS  : 200
HEADERS :
{'Accept-Ranges': 'bytes',
 'Cache-Control': 'public, max-age=0',
 'Connection': 'keep-alive',
 'Content-Length': '8688',
 'Content-Type': 'text/html; charset=UTF-8',
 'Date': 'Sun, 26 Apr 2026 13:58:26 GMT',
 'ETag': 'W/"21f0-15af0a320b8"',
 'Last-Modified': 'Tue, 21 Mar 2017 11:31:47 GMT',
 'X-Powered-By': 'Express'}

BODY    :
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="utf-8">
    <meta name="robots" content="all,follow">
    <meta name="googlebot" content="index,follow,snippet,archive">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="description" content="WeaveSocks Demo App">
    <meta name="author" content="Ondrej Svestka | ondrejsvestka.cz">
    <meta name="keywords" content="">

    <title>
        WeaveSocks
    </title>

    <meta name="keywords" content="">

    <link href='https://fonts.googleapis.com/css?family=Roboto:400,500,700,300,100'
          rel='stylesheet' type='text/css'>

    <!--

In [7]:
# ============================================================
# CÉLULA 7 — Página de categoria  (GET /category.html)
# ============================================================

resp = session.get(f"{BASE_URL}/category.html", timeout=TIMEOUT)
show(resp)
print(f"\n✓ Category page: {resp.status_code}")

STATUS  : 200
HEADERS :
{'Accept-Ranges': 'bytes',
 'Cache-Control': 'public, max-age=0',
 'Connection': 'keep-alive',
 'Content-Length': '11518',
 'Content-Type': 'text/html; charset=UTF-8',
 'Date': 'Sun, 26 Apr 2026 13:58:26 GMT',
 'ETag': 'W/"2cfe-15af0a320b8"',
 'Last-Modified': 'Tue, 21 Mar 2017 11:31:47 GMT',
 'X-Powered-By': 'Express'}

BODY    :
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="utf-8">
    <meta name="robots" content="all,follow">
    <meta name="googlebot" content="index,follow,snippet,archive">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="description" content="WeaveSocks Demo App">
    <meta name="author" content="Ondrej Svestka | ondrejsvestka.cz">
    <meta name="keywords" content="">

    <title>
        WeaveSocks
    </title>

    <meta name="keywords" content="">

    <link href='https://fonts.googleapis.com/css?family=Roboto:400,500,700,300,100'
          rel='stylesheet' type='text/css'>

    <!-

In [8]:
# ============================================================
# CÉLULA 8 — Consultar catálogo e escolher produto  (GET /catalogue)
# ============================================================

resp = session.get(f"{BASE_URL}/catalogue?size=9", timeout=TIMEOUT)
show(resp)

if resp.status_code == 200:
    items = resp.json()
    if isinstance(items, list) and items:
        chosen = random.choice(items)
        state["item_id"] = chosen.get("id")
        print(f"\n✓ Catálogo OK — {len(items)} produto(s)")
        print(f"  Produto escolhido : {chosen.get('name')!r}")
        print(f"  item_id           : {state['item_id']}")
    else:
        print("\n⚠ Catálogo vazio")
else:
    print("\n✗ Catálogo falhou")

STATUS  : 200
HEADERS :
{'Connection': 'keep-alive',
 'Date': 'Sun, 26 Apr 2026 13:58:26 GMT',
 'Transfer-Encoding': 'chunked',
 'X-Powered-By': 'Express',
 'set-cookie': 'md.sid=s%3ASj-avTO3JG35xG-75JLHmVUWmLbMRBvs.RBirivvi6txI2JcUz1ambZkJV4KP5mAtY5tK4J%2BD5ok; '
               'Path=/; HttpOnly'}

BODY    :
[{'count': 1,
  'description': 'Socks fit for a Messiah. You too can experience walking in '
                 'water with these special edition beauties. Each hole is '
                 'lovingly proggled to leave smooth edges. The only sock '
                 'approved by a higher power.',
  'id': '03fef6ac-1896-4ce8-bd69-b798f85c6e0b',
  'imageUrl': ['/catalogue/images/holy_1.jpeg',
               '/catalogue/images/holy_2.jpeg'],
  'name': 'Holy',
  'price': 99.99,
  'tag': ['magic', 'action']},
 {'count': 438,
  'description': 'proident occaecat irure et excepteur labore minim nisi amet '
                 'irure',
  'id': '3395a43e-2d88-40de-b95f-e00e1502085b',
  'imageUrl': [

In [9]:
# ============================================================
# CÉLULA 9 — Limpar carrinho  (DELETE /cart)
# ============================================================
sleep(0.5)
resp = session.delete(f"{BASE_URL}/cart", timeout=TIMEOUT)
show(resp)
print(f"\n✓ Carrinho limpo: {resp.status_code}")

STATUS  : 202
HEADERS :
{'Connection': 'keep-alive',
 'Date': 'Sun, 26 Apr 2026 13:58:27 GMT',
 'Transfer-Encoding': 'chunked',
 'X-Powered-By': 'Express',
 'set-cookie': 'md.sid=s%3AarkijUVkQhKFsx9x-qj9ZxL2zpIK2YfG.1uJOATg8yZm0UbJn6VAEM9QwpZgGwFnctGQWpBgvlpg; '
               'Path=/; HttpOnly'}

BODY    :


✓ Carrinho limpo: 202


In [10]:
# ============================================================
# CÉLULA 10 — Adicionar produto ao carrinho  (POST /cart)
# ============================================================
sleep(0.5)
if not state["item_id"]:
    print("⚠ Nenhum item selecionado — execute a Célula 8 primeiro.")
else:
    payload = {"id": state["item_id"], "quantity": 1}
    print(f"Payload: {payload}\n")

    resp = session.post(f"{BASE_URL}/cart", json=payload, timeout=TIMEOUT)
    show(resp)

    if resp.status_code in (200, 201, 202):
        print("\n✓ Item adicionado ao carrinho")
    else:
        print("\n✗ Falha ao adicionar ao carrinho")

Payload: {'id': 'd3588630-ad8e-49df-bbd7-3167f7efb246', 'quantity': 1}

STATUS  : 201
HEADERS :
{'Connection': 'keep-alive',
 'Date': 'Sun, 26 Apr 2026 13:58:27 GMT',
 'Transfer-Encoding': 'chunked',
 'X-Powered-By': 'Express',
 'set-cookie': 'md.sid=s%3AGIzyjYqY1oMiXTD-MXcNFclTSlJxZVN0.2MW31W9G8byHdWkKstJUT2yN06trxQW%2BdjJcI2vz9Zg; '
               'Path=/; HttpOnly'}

BODY    :


✓ Item adicionado ao carrinho


In [11]:
# ============================================================
# CÉLULA 11 — Ver carrinho  (GET /basket.html)
# ============================================================
sleep(0.5)
resp = session.get(f"{BASE_URL}/basket.html", timeout=TIMEOUT)
show(resp)
print(f"\n✓ Basket page: {resp.status_code}")

STATUS  : 200
HEADERS :
{'Accept-Ranges': 'bytes',
 'Cache-Control': 'public, max-age=0',
 'Connection': 'keep-alive',
 'Content-Length': '23020',
 'Content-Type': 'text/html; charset=UTF-8',
 'Date': 'Sun, 26 Apr 2026 13:58:28 GMT',
 'ETag': 'W/"59ec-15af0a320b8"',
 'Last-Modified': 'Tue, 21 Mar 2017 11:31:47 GMT',
 'X-Powered-By': 'Express'}

BODY    :
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="utf-8">
    <meta name="robots" content="all,follow">
    <meta name="googlebot" content="index,follow,snippet,archive">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="description" content="WeaveSocks Demo App">
    <meta name="author" content="Ondrej Svestka | ondrejsvestka.cz">
    <meta name="keywords" content="">

    <title>
        WeaveSocks
    </title>

    <meta name="keywords" content="">

    <link href='https://fonts.googleapis.com/css?family=Roboto:400,500,700,300,100'
          rel='stylesheet' type='text/css'>

    <!-

In [12]:
# ============================================================
# CÉLULA 12 — Criar endereço de entrega  (POST /addresses)
# ============================================================
# O front-end injeta req.body.userID = helpers.getCustomerId(req).
# getCustomerId retorna req.session.customerId quando logged_in existe.
# Se a sessão não tiver customerId, userID fica vazio e o serviço
# "user" cria o endereço sem associá-lo a ninguém.

payload = {
    "number":   str(random.randint(10, 9999)),
    "street":   "Rua Teste",
    "city":     "Sao Paulo",
    "postcode": "01001-000",
    "country":  "Brazil",
}
print(f"Payload: {payload}\n")

resp = session.post(f"{BASE_URL}/addresses", json=payload, timeout=TIMEOUT)
show(resp)

if resp.status_code in (200, 201) and not response_has_app_error(resp):
    data = resp.json()
    state["address_id"] = extract_object_id(data, link_rel="address")
    print(f"\n✓ Endereço criado — address_id: {state['address_id']}")

    # Verificação imediata: se GET /address retornar {"status_code": 500}
    # o endereço foi criado como órfão (userID estava vazio na hora do POST).
    sleep(0.5)
    chk = session.get(f"{BASE_URL}/address", timeout=TIMEOUT)
    if is_ok_response(chk):
        print("✓ Endereço recuperável via GET /address — associação OK")
    else:
        print("✗ GET /address retornou status_code 500 — endereço criado sem dono!")
        print("  Causa: req.session.customerId estava vazio no momento do POST.")
        print("  → Verifique a Célula 14 (diagnóstico de sessão) e reinicie.")
else:
    print("\n✗ Falha ao criar endereço")

Payload: {'number': '6643', 'street': 'Rua Teste', 'city': 'Sao Paulo', 'postcode': '01001-000', 'country': 'Brazil'}

STATUS  : 200
HEADERS :
{'Connection': 'keep-alive',
 'Date': 'Sun, 26 Apr 2026 13:58:28 GMT',
 'Transfer-Encoding': 'chunked',
 'X-Powered-By': 'Express',
 'set-cookie': 'md.sid=s%3ALrLMri4aqnpUfhExpyfnSyqGJ4h-UevF.MSxHXeyFosZUf1AzjmnlBcWP82FEPGxZMQqK6SzAAiE; '
               'Path=/; HttpOnly'}

BODY    :
{'id': '69ee1a044a863f0001768b03'}

✓ Endereço criado — address_id: 69ee1a044a863f0001768b03
✗ GET /address retornou status_code 500 — endereço criado sem dono!
  Causa: req.session.customerId estava vazio no momento do POST.
  → Verifique a Célula 14 (diagnóstico de sessão) e reinicie.


In [13]:
# ============================================================
# CÉLULA 13 — Criar cartão de crédito  (POST /cards)
# ============================================================
sleep(0.5)
payload = {
    "longNum": "5555555555554444",
    "expires": "12/30",
    "ccv":     "123",
}
print(f"Payload: {payload}\n")

resp = session.post(f"{BASE_URL}/cards", json=payload, timeout=TIMEOUT)
show(resp)

if resp.status_code in (200, 201) and not response_has_app_error(resp):
    data = resp.json()
    state["card_id"] = extract_object_id(data, link_rel="card")
    print(f"\n✓ Cartão criado — card_id: {state['card_id']}")

    # Mesma verificação imediata do endereço
    sleep(0.5)
    chk = session.get(f"{BASE_URL}/card", timeout=TIMEOUT)
    if is_ok_response(chk):
        print("✓ Cartão recuperável via GET /card — associação OK")
    else:
        print("✗ GET /card retornou status_code 500 — cartão criado sem dono!")
        print("  Causa: req.session.customerId estava vazio no momento do POST.")
        print("  → Verifique a Célula 14 (diagnóstico de sessão) e reinicie.")
else:
    print("\n✗ Falha ao criar cartão")

Payload: {'longNum': '5555555555554444', 'expires': '12/30', 'ccv': '123'}

STATUS  : 200
HEADERS :
{'Connection': 'keep-alive',
 'Date': 'Sun, 26 Apr 2026 13:58:29 GMT',
 'Transfer-Encoding': 'chunked',
 'X-Powered-By': 'Express',
 'set-cookie': 'md.sid=s%3AfvMH-t6vSmYCsWsnXHR2_ppllnjEFGMo.EuNzo2q4lqbdDemj2YXWA8uaf5YIupmAdGlIdWIDsRU; '
               'Path=/; HttpOnly'}

BODY    :
{'id': '69ee1a054a863f0001768b04'}

✓ Cartão criado — card_id: 69ee1a054a863f0001768b04
✗ GET /card retornou status_code 500 — cartão criado sem dono!
  Causa: req.session.customerId estava vazio no momento do POST.
  → Verifique a Célula 14 (diagnóstico de sessão) e reinicie.


In [14]:
# ============================================================
# CÉLULA 14 — Diagnóstico de sessão
# ============================================================
# Verifica se a sessão está íntegra antes do checkout.
# Se md.sid e logged_in não corresponderem, o checkout vai falhar.
sleep(0.5)
print("=== Estado dos cookies ===")
for c in session.cookies:
    print(f"  {c.name:20} = {c.value!r}")
    print(f"  {'':20}   domain={c.domain!r}  path={c.path!r}  expires={c.expires}")

print("\n=== Consistência de sessão ===")
sessao_ok = check_session_cookies()

print("\n=== Estado acumulado ===")
print(json.dumps(state, indent=2))

print("\n=== Resumo ===")
checks = {
    "customer_id válido":  is_valid_object_id(state.get("customer_id")),
    "item_id definido":    bool(state.get("item_id")),
    "address_id válido":   is_valid_object_id(state.get("address_id")),
    "card_id válido":      is_valid_object_id(state.get("card_id")),
    "sessão consistente":  sessao_ok,
}
for label, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  {status} {label}")

if all(checks.values()):
    print("\n✓ Tudo pronto para o checkout")
else:
    falhas = [k for k, v in checks.items() if not v]
    print(f"\n⚠ Falhas detectadas: {falhas}")
    print("  Corrija antes de prosseguir com as Células 15 e 16.")

=== Estado dos cookies ===
  logged_in            = 'SmPehyI1H8pB01injoAAe2f5qRA6VFe6'
                         domain='localhost.local'  path='/'  expires=1777215505
  md.sid               = 's%3AQ52QpUHFRiw4CicxVin5wTMfjGgt4Dld.7kBohw0Qc85JUEkyYBY0oGhgx1%2FPU%2FViQsgGUmlHR0U'
                         domain='localhost.local'  path='/'  expires=None

=== Consistência de sessão ===
md.sid (raw) : 's%3AQ52QpUHFRiw4CicxVin5wTMfjGgt4Dld.7kBohw0Qc85JUEkyYBY0oGhgx1%2FPU%2FViQsgGUmlHR0U'
logged_in    : 'SmPehyI1H8pB01injoAAe2f5qRA6VFe6'
session ID   : 'Q52QpUHFRiw4CicxVin5wTMfjGgt4Dld'
✗ PROBLEMA: md.sid e logged_in NÃO correspondem!
  → customerId não estará na sessão ativa
  → POST /addresses vai criar endereço SEM dono (userID vazio)

=== Estado acumulado ===
{
  "username": "user_fdc45c1b",
  "password": "Passw0rd!",
  "email": "user_fdc45c1b@example.com",
  "customer_id": "69ee1a004a863f0001768b02",
  "item_id": "d3588630-ad8e-49df-bbd7-3167f7efb246",
  "address_id": "69ee1a044a863f0001

In [15]:
# ============================================================
# CÉLULA 15 — Consultar endereço e cartão padrão
#              (GET /address  +  GET /card)
#
# O front-end retorna HTTP 200 com body {"status_code": 500} quando
# o usuário não tem endereço/cartão — por isso is_ok_response() verifica
# o body, não apenas o código HTTP.
# ============================================================
sleep(0.5)
resp_addr = session.get(f"{BASE_URL}/address", timeout=TIMEOUT)
print("── GET /address ──")
show(resp_addr)
has_default_address = is_ok_response(resp_addr)
print(f"\n→ endereço padrão OK: {has_default_address}")

print()
sleep(0.5)

resp_card = session.get(f"{BASE_URL}/card", timeout=TIMEOUT)
print("── GET /card ──")
show(resp_card)
has_default_card = is_ok_response(resp_card)
print(f"\n→ cartão padrão OK: {has_default_card}")

print(f"\nendereço padrão OK : {has_default_address}")
print(f"cartão padrão OK   : {has_default_card}")

if not has_default_address:
    print("\n⚠ Sem endereço padrão — execute a Célula 12.")
if not has_default_card:
    print("\n⚠ Sem cartão padrão — execute a Célula 13.")

── GET /address ──
STATUS  : 200
HEADERS :
{'Connection': 'keep-alive',
 'Date': 'Sun, 26 Apr 2026 13:58:30 GMT',
 'Transfer-Encoding': 'chunked',
 'X-Powered-By': 'Express',
 'set-cookie': 'md.sid=s%3AzqxHv-yFCQrV7lCbH0N-gzUJ9nwvQgfY.%2B%2FL7YVaJAbxiumRaAFUIhSifPym%2FzzvS6I0V%2FY1Jrzs; '
               'Path=/; HttpOnly'}

BODY    :
{'status_code': 500}

→ endereço padrão OK: False

── GET /card ──
STATUS  : 200
HEADERS :
{'Connection': 'keep-alive',
 'Date': 'Sun, 26 Apr 2026 13:58:31 GMT',
 'Transfer-Encoding': 'chunked',
 'X-Powered-By': 'Express',
 'set-cookie': 'md.sid=s%3As0-k_OOymOryTMLmFrsD52PM5DnRNyWk.pekXqAXlRInoWG4fgT4K16Z4QRIe7V4dkCu%2BE6PddwA; '
               'Path=/; HttpOnly'}

BODY    :
{'status_code': 500}

→ cartão padrão OK: False

endereço padrão OK : False
cartão padrão OK   : False

⚠ Sem endereço padrão — execute a Célula 12.

⚠ Sem cartão padrão — execute a Célula 13.


In [16]:
# ============================================================
# CÉLULA 16 — Finalizar pedido  (POST /orders)
# ============================================================
sleep(0.5)
if not has_default_address or not has_default_card:
    print("⚠ Endereço ou cartão padrão ausente — execute as Células 12–15.")
elif not state["item_id"]:
    print("⚠ Carrinho vazio — execute as Células 8–10 primeiro.")
else:
    # Não envia body: o front-end usa customerId da sessão HTTP
    resp = session.post(f"{BASE_URL}/orders", timeout=TIMEOUT)
    show(resp)

    if resp.status_code in (200, 201, 202) and not response_has_app_error(resp):
        print("\n✓ Pedido realizado com sucesso!")
    else:
        print("\n✗ Checkout falhou")

⚠ Endereço ou cartão padrão ausente — execute as Células 12–15.


In [17]:
# ============================================================
# CÉLULA 17 — Ver pedidos  (GET /customer-orders.html)
# ============================================================
sleep(0.5)
resp = session.get(f"{BASE_URL}/customer-orders.html", timeout=TIMEOUT)
show(resp)
print(f"\n✓ Orders page: {resp.status_code}")

STATUS  : 200
HEADERS :
{'Accept-Ranges': 'bytes',
 'Cache-Control': 'public, max-age=0',
 'Connection': 'keep-alive',
 'Content-Length': '6368',
 'Content-Type': 'text/html; charset=UTF-8',
 'Date': 'Sun, 26 Apr 2026 13:58:32 GMT',
 'ETag': 'W/"18e0-15af0a320b8"',
 'Last-Modified': 'Tue, 21 Mar 2017 11:31:47 GMT',
 'X-Powered-By': 'Express'}

BODY    :
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="utf-8">
    <meta name="robots" content="all,follow">
    <meta name="googlebot" content="index,follow,snippet,archive">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="description" content="WeaveSocks Demo App">
    <meta name="author" content="Ondrej Svestka | ondrejsvestka.cz">
    <meta name="keywords" content="">

    <title>
        WeaveSocks
    </title>

    <meta name="keywords" content="">

    <link href='https://fonts.googleapis.com/css?family=Roboto:400,500,700,300,100'
          rel='stylesheet' type='text/css'>

    <!--

In [18]:
# ============================================================
# CÉLULA 18 — Estado final da sessão
# ============================================================

print("Estado do usuário virtual:")
print(json.dumps(state, indent=2))

print("\nCookies da sessão:")
for name, value in session.cookies.items():
    print(f"  {name} = {value}")

print("\nVerificação final de sessão:")
check_session_cookies()

Estado do usuário virtual:
{
  "username": "user_fdc45c1b",
  "password": "Passw0rd!",
  "email": "user_fdc45c1b@example.com",
  "customer_id": "69ee1a004a863f0001768b02",
  "item_id": "d3588630-ad8e-49df-bbd7-3167f7efb246",
  "address_id": "69ee1a044a863f0001768b03",
  "card_id": "69ee1a054a863f0001768b04"
}

Cookies da sessão:
  logged_in = SmPehyI1H8pB01injoAAe2f5qRA6VFe6
  md.sid = s%3As0-k_OOymOryTMLmFrsD52PM5DnRNyWk.pekXqAXlRInoWG4fgT4K16Z4QRIe7V4dkCu%2BE6PddwA

Verificação final de sessão:
md.sid (raw) : 's%3As0-k_OOymOryTMLmFrsD52PM5DnRNyWk.pekXqAXlRInoWG4fgT4K16Z4QRIe7V4dkCu%2BE6PddwA'
logged_in    : 'SmPehyI1H8pB01injoAAe2f5qRA6VFe6'
session ID   : 's0-k_OOymOryTMLmFrsD52PM5DnRNyWk'
✗ PROBLEMA: md.sid e logged_in NÃO correspondem!
  → customerId não estará na sessão ativa
  → POST /addresses vai criar endereço SEM dono (userID vazio)


False